In [3]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import Literal, TypedDict
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [4]:
load_dotenv()

True

In [5]:
LLM = ChatGroq(model = "openai/gpt-oss-120b", temperature=0.5)

In [6]:
class Sentimentschema(BaseModel):
    
    sentiment: Literal["positive", "negative"] = Field(description="sentiment of the review either positive or negative")

In [7]:
structured_model = LLM.with_structured_output(Sentimentschema)

In [8]:
# test
prompt = "what is the sentiment of the follwoing statement - This product is bad"

structured_model.invoke(prompt)

Sentimentschema(sentiment='negative')

In [9]:
class SentimentState(TypedDict):
    
    review: str
    sentiment: Literal['positive', 'negative']
    diagonsis: dict
    response: str

In [14]:
def find_senti(state: SentimentState):
    prompt = f"find the sentiment of the review i got for my product - {state['review']}"
    sentiment = structured_model.invoke(prompt).sentiment
    
    return {'sentiment': sentiment}

In [15]:
# now lets create the graph
graph = StateGraph(SentimentState)

# node
graph.add_node('Find_sentiment', find_senti)

# edge
graph.add_edge(START, 'Find_sentiment')

workflow = graph.compile()

In [17]:
initial_state = {
    'review' : "The product is worst but has few good features"
}

final_state = workflow.invoke(initial_state)
print(final_state)

{'review': 'The product is worst but has few good features', 'sentiment': 'negative'}
